# COMPX525 A2 Synthetic BirdNET Features

Extract BirdNET embeddings for generated synthetic audio.


In [ ]:
from __future__ import annotations

import csv
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
INPUT_ROOT = Path("/kaggle/input")
BATCH_SIZE = 32
THREADS = 2

def run_cmd(cmd, check=True):
    print("CMD:", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"command failed with code {result.returncode}: {cmd}")
    return result

def ensure_birdnet():
    try:
        import birdnet_analyzer  # noqa
        import perch_hoplite  # noqa
        print("BirdNET dependencies already available")
        return
    except Exception as exc:
        print("BirdNET import check failed:", repr(exc))
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "birdnet-analyzer", "perch-hoplite==0.1.3"])
    import birdnet_analyzer  # noqa
    import perch_hoplite  # noqa
    print("BirdNET dependencies installed")

def read_manifest(path: Path) -> list[dict[str, str]]:
    with path.open(newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def discover_generation_outputs() -> tuple[list[dict[str, str]], dict[str, Path]]:
    manifest_rows = []
    audio_by_rel: dict[str, Path] = {}
    for manifest in sorted(INPUT_ROOT.rglob("synthetic_generation_manifest.csv")):
        chunk_root = manifest.parent
        print("Manifest:", manifest)
        for row in read_manifest(manifest):
            row = dict(row)
            row["source_manifest_path"] = str(manifest)
            manifest_rows.append(row)
            rel = row.get("relative_audio_path", "")
            candidate = chunk_root / rel
            if rel and candidate.exists():
                audio_by_rel[rel] = candidate
    return manifest_rows, audio_by_rel

def build_audio_tree(rows: list[dict[str, str]], audio_by_rel: dict[str, Path], out_root: Path) -> pd.DataFrame:
    if out_root.exists():
        shutil.rmtree(out_root)
    out_root.mkdir(parents=True, exist_ok=True)

    selected = []
    for row in rows:
        if row.get("filter_status") != "generated_pending_birdnet_filter":
            continue
        rel = row.get("relative_audio_path", "")
        source = audio_by_rel.get(rel)
        if source is None or not source.exists():
            row["birdnet_filter_status"] = "missing_audio"
            row["birdnet_filter_reason"] = "relative_audio_path not found in attached kernel outputs"
            selected.append(row)
            continue
        target = out_root / str(row["primary_label"]) / Path(rel).name
        target.parent.mkdir(parents=True, exist_ok=True)
        try:
            os.symlink(source, target)
        except OSError:
            shutil.copy2(source, target)
        row["birdnet_input_path"] = str(target)
        row["birdnet_filter_status"] = "prepared_for_embedding"
        row["birdnet_filter_reason"] = ""
        selected.append(row)
    return pd.DataFrame(selected)

def run_birdnet_embeddings(input_root: Path, database_root: Path, output_csv: Path) -> subprocess.CompletedProcess:
    if database_root.exists():
        shutil.rmtree(database_root)
    cmd = [
        sys.executable,
        "-m",
        "birdnet_analyzer.embeddings",
        "-db",
        database_root,
        "-i",
        input_root,
        "--file_output",
        output_csv,
        "--batch_size",
        str(BATCH_SIZE),
        "--threads",
        str(THREADS),
    ]
    start = time.time()
    result = run_cmd(cmd, check=False)
    print("BirdNET elapsed_seconds", round(time.time() - start, 3), "output_exists", output_csv.exists())
    return result

def parse_embedding_string(value: str) -> np.ndarray:
    return np.fromstring(str(value), sep=",", dtype=np.float32)

def aggregate_birdnet_embeddings(embedding_csv: Path, output_prefix: Path) -> dict:
    emb_df = pd.read_csv(embedding_csv)
    emb_df["primary_label"] = emb_df["file_path"].map(lambda x: Path(str(x)).parent.name)
    emb_df["audio_basename"] = emb_df["file_path"].map(lambda x: Path(str(x)).name)
    emb_df["filename"] = emb_df["primary_label"] + "/" + emb_df["audio_basename"]
    emb_df["synthetic_id"] = emb_df["audio_basename"].str.replace(".wav", "", regex=False)

    vectors = np.stack([parse_embedding_string(x) for x in emb_df["embedding"]])
    segment_meta = emb_df[["synthetic_id", "filename", "file_path", "start", "end", "primary_label"]].copy()
    segment_path = output_prefix.with_name(output_prefix.name + "_segments.csv")
    segment_meta.to_csv(segment_path, index=False)

    synthetic_ids = []
    filenames = []
    labels = []
    features = []
    for filename, idx in emb_df.groupby("filename").indices.items():
        group_vectors = vectors[np.array(idx)]
        synthetic_id = Path(filename).stem
        synthetic_ids.append(synthetic_id)
        filenames.append(filename)
        labels.append(str(filename).split("/", 1)[0])
        features.append(np.concatenate([group_vectors.mean(axis=0), group_vectors.std(axis=0), group_vectors.max(axis=0)]))

    feature_array = np.stack(features).astype(np.float32)
    metadata = pd.DataFrame({"synthetic_id": synthetic_ids, "filename": filenames, "primary_label": labels})
    metadata_path = output_prefix.with_name(output_prefix.name + "_clip_metadata.csv")
    feature_path = output_prefix.with_name(output_prefix.name + "_clip_features.npz")
    metadata.to_csv(metadata_path, index=False)
    np.savez_compressed(
        feature_path,
        features=feature_array,
        filenames=np.array(filenames),
        labels=np.array(labels),
        synthetic_ids=np.array(synthetic_ids),
    )
    return {
        "embedding_csv": str(embedding_csv),
        "segments": int(len(emb_df)),
        "clips": int(len(metadata)),
        "embedding_dim": int(vectors.shape[1]),
        "clip_feature_dim": int(feature_array.shape[1]),
        "segment_meta_path": str(segment_path),
        "metadata_path": str(metadata_path),
        "feature_path": str(feature_path),
    }

ensure_birdnet()
manifest_rows, audio_by_rel = discover_generation_outputs()
print("manifest_rows", len(manifest_rows), "audio_by_rel", len(audio_by_rel))

audio_root = WORK_DIR / "synthetic_birdnet_audio"
prepared_df = build_audio_tree(manifest_rows, audio_by_rel, audio_root)
prepared_df.to_csv(WORK_DIR / "synthetic_birdnet_prepared_manifest.csv", index=False)
print("prepared rows", len(prepared_df), "prepared audio", int((prepared_df["birdnet_filter_status"] == "prepared_for_embedding").sum()))

birdnet_db = WORK_DIR / "synthetic_birdnet_db"
birdnet_out = WORK_DIR / "synthetic_birdnet_embeddings.csv"
result = run_birdnet_embeddings(audio_root, birdnet_db, birdnet_out)
aggregation = aggregate_birdnet_embeddings(birdnet_out, WORK_DIR / "synthetic_birdnet") if birdnet_out.exists() else None

accepted_ids = set()
if aggregation:
    meta = pd.read_csv(aggregation["metadata_path"])
    accepted_ids = set(meta["synthetic_id"].astype(str))

prepared_df["birdnet_filter_status"] = prepared_df.apply(
    lambda row: "accepted_embedding_extracted" if str(row.get("synthetic_id")) in accepted_ids else row.get("birdnet_filter_status"),
    axis=1,
)
prepared_df["birdnet_filter_reason"] = prepared_df.apply(
    lambda row: "" if row.get("birdnet_filter_status") == "accepted_embedding_extracted" else row.get("birdnet_filter_reason", ""),
    axis=1,
)
prepared_df.to_csv(WORK_DIR / "synthetic_birdnet_filtered_manifest.csv", index=False)

summary = {
    "manifest_rows": int(len(manifest_rows)),
    "audio_files_discovered": int(len(audio_by_rel)),
    "prepared_rows": int(len(prepared_df)),
    "prepared_for_embedding": int((prepared_df["birdnet_filter_status"].isin(["prepared_for_embedding", "accepted_embedding_extracted"])).sum()),
    "accepted_embedding_extracted": int((prepared_df["birdnet_filter_status"] == "accepted_embedding_extracted").sum()),
    "birdnet_exit_code": int(result.returncode),
    "birdnet_output_exists": bool(birdnet_out.exists()),
    "aggregation": aggregation,
}
(WORK_DIR / "synthetic_birdnet_summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")
print(json.dumps(summary, indent=2))

if birdnet_out.exists():
    birdnet_out.unlink()
if birdnet_db.exists():
    shutil.rmtree(birdnet_db)
print("Synthetic BirdNET stage complete")
